# Week 1 — reproduction gate (protocol §A.4)

This notebook contains **no logic**. It imports from `src/` and calls one function
(CLAUDE.md §4). Anything you are tempted to write here belongs in `src/` where it
can be tested.

Attach two Kaggle Datasets before running:

| Dataset | Contents | Mounted at |
|---|---|---|
| `neuron-death-code` | this repository | `/kaggle/input/neuron-death-code` |
| `neuron-death-mnist` | `mnist.npz` from `scripts/prepare_data.py` | `/kaggle/input/neuron-death-mnist` |

Set the accelerator to **GPU T4 x2**. `/kaggle/working` does not persist: run the
last cell before the session ends, and push `runs.zip` to a versioned Dataset.

In [ ]:
import os, sys, glob, shutil, subprocess

REPO = "/kaggle/input/neuron-death-code"
DATA = "/kaggle/input/neuron-death-mnist"
RUNS = "/kaggle/working/runs"

# Must be set before any CUDA context exists (see src/config.set_determinism).
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
sys.path.insert(0, REPO)

import torch
print(torch.__version__, torch.cuda.device_count(), "GPU(s)")
print(sorted(os.listdir(DATA)))

## Always run the tests first

Twenty seconds of testing beats losing a 13-hour sweep (CLAUDE.md §8). This runs
on synthetic data and needs no GPU.

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "tests/"], cwd=REPO, check=True)

## One config (smoke test)

Point `data.root` at the mounted dataset. Expect ~3–6 minutes for 200 tasks.

In [ ]:
from src.config import load_config
from src.train import Trainer

cfg = load_config(f"{REPO}/configs/gate/gate_pmnist_w500_sgd_lr0p01_s0.json")
cfg["data"]["root"] = DATA
run_dir = Trainer(cfg, runs_root=RUNS).run()
print(run_dir)

## The full gate: 15 runs, two per GPU pass

`launch_pair.py` runs two configs at a time via `CUDA_VISIBLE_DEVICES` and stops
launching new work at `--budget-hours`, so the 12-hour kill never lands mid-run.
Anything interrupted resumes from its own checkpoint by `run_id`.

In [ ]:
configs = sorted(glob.glob(f"{REPO}/configs/gate/*.json"))
subprocess.run(
    [sys.executable, "scripts/launch_pair.py", *configs,
     "--runs-root", RUNS, "--budget-hours", "10.5"],
    cwd=REPO, env={**os.environ, "NEURON_DEATH_DATA": DATA}, check=True,
)

## Evaluate the gate

Applies the frozen criterion in `configs/analysis_plan.json`. If the accuracy
gate passes but dead units stay flat, **stop and report it** — that dissociation
is itself a result and changes the paper (protocol §A.4).

In [ ]:
subprocess.run(
    [sys.executable, "-m", "src.analysis.gate", "--runs-root", RUNS,
     "--pattern", "gate_*"],
    cwd=REPO, check=False,
)

## Persist results

`/kaggle/working` is wiped when the session ends. Run this, then add `runs.zip`
as a new version of the results Dataset, then update `runs/LEDGER.md` locally.

In [ ]:
subprocess.run([sys.executable, "scripts/update_ledger.py", "--runs-root", RUNS,
                "--out", "/kaggle/working/LEDGER.md"], cwd=REPO, check=True)
print(shutil.make_archive("/kaggle/working/runs", "zip", RUNS))